## 4) Compute & Cache DTW distances (query → 4 refs)

In [1]:
# 04_dtw_cache.ipynb (minimal)
import sys, os
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.dtw import compute_dtw as cd

# avoid BLAS oversubscription when using multiple processes
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

pairs_dir = project_root / "data" / "pairs"
cache_dir = project_root / "data" / "dtw_cache"

jobs = [
    ("dev",  "genuine"),
    ("dev",  "skilled"),
    ("dev",  "random"),
    ("test", "genuine"),
    ("test", "skilled"),
    ("test", "random"),
]

for split, case in jobs:
    in_path  = pairs_dir / split / f"pairs_{split}_{case}.parquet"
    out_path = cache_dir / split / f"dtw_cache_{split}_{case}.parquet"
    print(f"[{split}/{case}] {in_path} -> {out_path}")
    cd.build_cache(
        in_path,
        out_path,
        backend=None,   # auto: dtaidistance if available else python
        window=20,      # Sakoe–Chiba band
        n_jobs=None,    # or os.cpu_count()
        overwrite=False,
        show_progress=True,
    )
    df = pd.read_parquet(out_path, engine="pyarrow")
    print("  rows:", len(df))

[dev/genuine] c:\Users\mattt\Skripsie\Projects\DTW-project\data\pairs\dev\pairs_dev_genuine.parquet -> c:\Users\mattt\Skripsie\Projects\DTW-project\data\dtw_cache\dev\dtw_cache_dev_genuine.parquet


DTW pairwise | backend=dtaidistance | w=20 | procs=20: 100%|██████████| 14400/14400 [48:14<00:00,  4.97pair/s]   


  rows: 14400
[dev/skilled] c:\Users\mattt\Skripsie\Projects\DTW-project\data\pairs\dev\pairs_dev_skilled.parquet -> c:\Users\mattt\Skripsie\Projects\DTW-project\data\dtw_cache\dev\dtw_cache_dev_skilled.parquet


DTW pairwise | backend=dtaidistance | w=20 | procs=20: 100%|██████████| 14400/14400 [1:36:25<00:00,  2.49pair/s]  


  rows: 14400
[dev/random] c:\Users\mattt\Skripsie\Projects\DTW-project\data\pairs\dev\pairs_dev_random.parquet -> c:\Users\mattt\Skripsie\Projects\DTW-project\data\dtw_cache\dev\dtw_cache_dev_random.parquet


...

  rows: 4800
[test/skilled] c:\Users\mattt\Skripsie\Projects\DTW-project\data\pairs\test\pairs_test_skilled.parquet -> c:\Users\mattt\Skripsie\Projects\DTW-project\data\dtw_cache\test\dtw_cache_test_skilled.parquet


DTW pairwise | backend=dtaidistance | w=20 | procs=20: 100%|██████████| 4800/4800 [36:33<00:00,  2.19pair/s]  


  rows: 4800
[test/random] c:\Users\mattt\Skripsie\Projects\DTW-project\data\pairs\test\pairs_test_random.parquet -> c:\Users\mattt\Skripsie\Projects\DTW-project\data\dtw_cache\test\dtw_cache_test_random.parquet


DTW pairwise | backend=dtaidistance | w=20 | procs=20: 100%|██████████| 4800/4800 [14:01<00:00,  5.71pair/s]  


  rows: 4800


In [2]:
import pandas as pd, itertools as it

for split, case in jobs:
    df_pairs = pd.read_parquet(pairs_dir / split / f"pairs_{split}_{case}.parquet", engine="pyarrow")
    df_cache = pd.read_parquet(cache_dir / split / f"dtw_cache_{split}_{case}.parquet", engine="pyarrow")
    # 48 comparisons per user per case
    expected = 48 * (300 if split == "dev" else 100)
    assert len(df_pairs) == expected and len(df_cache) == expected
    # schema
    req = {"pair_id","label","d_raw","d_bound","path_len","len_A","len_B","backend","window","mode"}
    assert req.issubset(df_cache.columns)
print("✅ DTW cache files align with the protocol.")

✅ DTW cache files align with the protocol.
